# Query categories

1. **Factual retrieval**: direct retrieval of stored knowledge
   > What are the last four videos I have watched?
2. **Cross-modal retrieval**: requiring information from different modalities
   > Find videos that show a dog
3. **Analytical multi-hop synthesis**: combining multiple pieces of evidence
   > Considering the last twenty videos I watched, is there any bias that comes up?
4. **Conversational follow-up/personalised context**: memory-sensitive queries
   > What was that fourth video's thumbnail, again?

# Architectures
| Ablation | Short-Term Memory | KB | Vector DB Indexing | Bias Analysis Sub-Agent |
|----------|:-----------------:|:--:|:------------------:|:-----------------------:|
| 0. **Vanilla**        | Y | N/A                   | N/A    | N    |
| 1. **Text RAG**       | Y | CSV                   | Text   | N    |
| 2. **MM-RAG**         | Y | CSV + Thumbnails      | Hybrid | N    |
| 3. **Agentic MM-RAG** | Y | CSV + Thumbnails      | Hybrid | Y    |


## Excluded architectures
The following architectures were not considered, as they would have been less informative:
1. **No memory** - Predictable context benefit, complex multi-turn benchmark needed
2. **Hybrid indexing + text retrieval** - Adds complexity, core multimodal story already proven 
3. **Router-based** - Simple queries don't differentiate, complex ones hard to score objectively
4. **VLM-based agent** - Tests model choice not retrieval design
5. **Captioning vs direct embedding** - Engineering detail, not system architecture

# Creating the dataset
The task of identifying biases in a random user's history presents two main issues:
1. The dataset contains random user histories, which may or may not contain biases for the model to work on.
2. The problem doesn't have such thing as "ground truth" in the unambiguous sense of "correct sets of biases". The definition of "bias" is deliberately left open-ended.

The following solutions are proposed to address these issues:
1. Augment the dataset with synthetic data that clearly contains biases.
2. Use a teacher/student approach to create a pseudo-ground-truth.

In [7]:
from langchain_groq import ChatGroq
from pydantic import SecretStr
from pydantic import BaseModel
from typing import Dict, List
import pandas as pd
import string
import random

with open('../llm-api-key.txt') as f:
    api_key = SecretStr(f.readline().strip())

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    reasoning_effort="medium",
    api_key=api_key,
    temperature=0.6,
)

## Augmenting the dataset

In [ ]:
df = pd.read_csv("../knowledge/clean_youtube_watch_log.csv")
n = df.shape[0]
bias_n = int(n * 0.3)

print(f"Original: {n} rows")
print(f"Augmented: {bias_n} biased rows ({bias_n/n*100:.1f}%)")
print(f"Total: {n + bias_n} rows")

format = ",".join(df.columns.tolist())
example = ",".join(map(str, df.iloc[6].tolist()))

Original: 987 rows
Augmented: 296 biased rows (30.0%)
Total: 1283 rows


'log_id,user_id,video_id,watch_date,subscribed,playlist_name,video_title'

In [22]:
prompt = """
Generate 50 rows to augment an existing YouTube watch log dataset, introducing detectable biases in content consumption. 
Each row should represent a video watched by a user with varied timestamps, realistic thumbnail URLs, and durations between 5-45 minutes.

# Bias Categories & Title Examples:
- Rightist Extremism:, "Why Liberals Destroy Everything", "The War on Western Civilization", "Globalist Agenda EXPOSED" and similar
- Leftist Extremism: "Why Capitalism Must End", "Fascism Returns: The MAGA Threat", "Corporate Greed is Killing Us" and similar
- Conspiracy (sensational framing): "COVID Vaccine Hoax Finally Revealed", "Election 2020: The Stolen Truth", "Climate Change: The Biggest Scam" and similar
- Sensational/Clickbait: "SHOCKING: What They Hide From You", "You Won't Believe This Election Fraud", "EXPOSED: The 5G Danger Nobody Talks About" and similar
- Polarization: "Us vs Them: The Culture War", "Red vs Blue: America Divided", "The Great Divide: Why We Can't Unite" and similar
- and more...

# Output Format:
log_id,user_id,video_id,watch_date,subscribed,playlist_name,video_title

# Rules:
- log_id = unique random integer with 6 digits
- user_id = always 25 fixed
- video_id = unique random string of 11 characters
- watch_date = realistic timestamps between 2014-04-12 06:38:03 and 2018-09-11 17:02:26
- subscribed = always None
- playlist_name = always None
- video_title = realistic video titles referring to one or more of the above biases (IMPORTANT)

# Example row:
146533,25,Q5JnbpltAAo,2018-09-10 22:59:44,None,None,The Evil Jungle | Ben 10 | Cartoon Network

Give me a final recap of what you generated, with a breakdown of some of the biased rows produced.
"""

response = llm.invoke(prompt)

###  Double-check the augmented dataset before merge

In [25]:
df = pd.read_csv("augment.csv")

In [26]:
# Define the date range
start_date = "2014-04-12 06:38:03"
end_date = "2018-09-11 17:02:26"

# Check for violations and print them
if not df.iloc[:, 0].is_unique:
    print("First column has duplicate values:")
    print(df[df.iloc[:, 0].duplicated(keep=False)])

if not (df.iloc[:, 1] == 25).all():
    print("Second column has values other than 25:")
    print(df[df.iloc[:, 1] != 25])

if not df.iloc[:, 2].apply(lambda x: isinstance(x, str) and len(x) == 11).all():
    print("Third column has values that are not 11-character strings:")
    print(df[~df.iloc[:, 2].apply(lambda x: isinstance(x, str) and len(x) == 11)])

if not df.iloc[:, 2].is_unique:
    print("Third column has duplicate values:")
    print(df[df.iloc[:, 2].duplicated(keep=False)])

if not df.iloc[:, 3].between(start_date, end_date).all():
    print("Fourth column has dates outside the valid range:")
    print(df[~df.iloc[:, 3].between(start_date, end_date)])

In [19]:
# Fix third column to ensure uniqueness and 11-character length
existing = set(df.iloc[:, 2])
df.iloc[:, 2] = df.iloc[:, 2].apply(
    lambda x: x if len(x) == 11 and x not in existing else ''.join(random.choices(string.ascii_letters + string.digits, k=11))
)
df.to_csv("augment.csv", index=False)

### Merging the two datasets

In [31]:
# Load the second dataset
df2 = pd.read_csv("../knowledge/clean_youtube_watch_log.csv")

# Merge the two datasets
df_merged = pd.concat([df, df2], ignore_index=True)

# Shuffle the rows
df_merged = df_merged.sample(frac=1).reset_index(drop=True)

### Double-check the merged dataset

In [ ]:
# Define the date range
start_date = "2014-04-12 06:38:03"
end_date = "2018-09-11 17:02:26"

# Check for violations and print them
if not df_merged.iloc[:, 0].is_unique:
    print("First column has duplicate values:")
    print(df_merged[df_merged.iloc[:, 0].duplicated(keep=False)])

if not (df_merged.iloc[:, 1] == 25).all():
    print("Second column has values other than 25:")
    print(df_merged[df_merged.iloc[:, 1] != 25])

duplicates = df_merged[df_merged.duplicated(subset=['video_id'], keep=False)]
differing_titles = duplicates.groupby('video_id').filter(
    lambda group: group.iloc[:, 6].nunique() > 1  # Check if 'video_title' has more than one unique value
)

if not differing_titles.empty:
    print("Third column has duplicate video_id with different video_title:")
    print(differing_titles)

if not df_merged.iloc[:, 3].between(start_date, end_date).all():
    print("Fourth column has dates outside the valid range:")
    print(df_merged[~df_merged.iloc[:, 3].between(start_date, end_date)])

In [ ]:
df_merged.to_csv("augmented_clean_youtube_watch_log.csv", index=False)

## Generating the questions

In [ ]:
class StructuredOutput(BaseModel):
    factual: List[Dict[str, str]]
    cross_modal: List[Dict[str, str]]
    analytical: List[Dict[str, str]]
    conversational: List[Dict[str, str]]

prompt = """
You are an expert evaluator for multimodal agent systems. 
Your goal is to generate specific test queries for a YouTube watch history bias analysis agent.

# Query Categories 
Your goal is to generate 4 queries for each of these categories:
1. **Factual retrieval**: Direct lookup from watch history CSV
2. **Cross-modal retrieval**: Given a title or a thumbnail (assume you have its url), find a similar video in the history
3. **Analytical multi-hop synthesis**: Detect a bias pattern across multiple videos (e.g., political leaning, sensationalism)
4. **Conversational follow-up**: Memory-dependent queries referencing prior responses

# Architectures
Your goal is to generate queries that can effectively benchmark the answering quality of these agent architectures:
1. Vanilla: No KB/retrieval
2. Text RAG: CSV + text-only indexing  
3. MM-RAG: CSV + thumbnails + hybrid indexing
4. Agentic: MM-RAG + bias analysis sub-agent

# Output Format
{
    "factual": [
        { "[question]": "[golden answer]" },
        ...
    ],
    "cross_modal": [
        { "[question]": "[golden answer]" },
        ...
    ],
    "analytical": [
        { "[question]": "[golden answer]" },
        ...
    ],
    "conversational": [
        { "[question]": "[golden answer]" },
        ...
    ]
}

# Rules
- Make queries realistic for YouTube watch history (channels, thumbnails, timestamps)
- Conversational queries MUST reference specific prior query results 
- Cross-modal queries should clearly need thumbnail similarity
- Analytical queries should require pattern detection across multiple videos
- No tool calls - pure query generation only
"""

response = llm \
    .with_structured_output(StructuredOutput, method="json_schema") \
    .invoke(prompt)

with open("results.json", "w") as f:
    f.write(response.model_dump_json()) # type: ignore